# 学习曲线：诊断过拟合与欠拟合
> 难度标签：中级 | 预计时长：10-20分钟 | 前置知识：无学习经验


> 所属模块：12_模型评估与选择 | 源文件：学习曲线.py | 核心功能：训练/验证曲线、偏差-方差分析

## 概述

学习曲线画出训练集大小 vs 训练/验证分数的曲线，帮助诊断模型是过拟合还是欠拟合，以及增加数据是否有帮助。

**导入必要的库**

In [ ]:
# 确保中文输出正常（Windows 环境兼容）
import sys
try:
    sys.stdout.reconfigure(encoding='utf-8')
except AttributeError:
    pass

import numpy as np
from sklearn.datasets import make_classification
from sklearn.model_selection import learning_curve, validation_curve
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
# --- 导入库 ---
from sklearn.tree import DecisionTreeClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline

# 生成数据
X, y = make_classification(
    n_samples=500, n_features=15, n_informative=8,
    n_classes=2, random_state=42
)

## 数学原理

### 1. 学习曲线（Learning Curve）

学习曲线展示**训练集大小** $m$ 与模型性能的关系：

$$\text{train}(m) = \frac{1}{m}\sum_{i=1}^{m} L(y_i, f_m(x_i))$$

$$\text{val}(m) = \frac{1}{|D_{val}|}\sum_{(x,y) \in D_{val}} L(y, f_m(x))$$

其中 $f_m$ 是在 $m$ 个样本上训练的模型。

### 2. 偏差-方差分解

模型的期望泛化误差可分解为：

$$\mathbb{E}[L] = \text{Bias}^2 + \text{Var} + \text{Noise}$$

- **偏差（Bias）**：$\mathbb{E}[f_m(x)] - f^*(x)$，模型的系统性误差
- **方差（Var）**：$\text{Var}[f_m(x)]$，模型对训练数据的敏感度
- **噪声（Noise）**：数据本身的不可约误差

### 3. 学习曲线的典型模式

**欠拟合**（高偏差）：
- 训练分数和验证分数都很低
- 增加训练数据几乎无改善
- 两条曲线趋于收敛，但收敛值很低

$$\lim_{m \to \infty} \text{val}(m) - \text{train}(m) \approx 0, \quad \text{但分数都低}$$

**过拟合**（高方差）：
- 训练分数很高，验证分数较低
- 增加训练数据能逐渐改善（两曲线趋向接近）

$$\text{train}(m) \approx 1, \quad \text{val}(m) \ll 1, \quad \text{gap} = \text{train} - \text{val} \text{ 大}$$

### 4. 验证曲线（Validation Curve）

验证曲线展示**超参数值**与模型性能的关系：

$$\text{train}(\lambda), \quad \text{val}(\lambda)$$

- $\lambda$ 小（模型复杂）：$\text{train}$ 高，$\text{val}$ 可能低（过拟合）
- $\lambda$ 大（模型简单）：$\text{train}$ 低，$\text{val}$ 也低（欠拟合）
- 最优 $\lambda^*$：$\text{val}(\lambda)$ 最大

代码中 `validation_curve` 改变 `max_depth` 观察训练/验证分数变化。

### 5. 偏差-方差与模型复杂度

$$\text{总误差}(\text{复杂度}) = \text{Bias}^2(\text{复杂度}) + \text{Var}(\text{复杂度}) + \text{Noise}$$

- 低复杂度：高偏差、低方差
- 高复杂度：低偏差、高方差
- 最优复杂度：总误差最小的平衡点

### 6. 数据量与泛化

学习曲线的收敛行为：

$$\text{val}(m) - \text{val}(\infty) \propto \frac{1}{\sqrt{m}}$$

验证误差以 $O(1/\sqrt{m})$ 的速率收敛。如果两曲线仍有较大间距，增加数据可能有帮助。

### 7. 代码与数学的对应

| 代码 | 数学含义 |
|------|----------|
| `learning_curve(model, X, y, train_sizes=...)` | 计算不同 $m$ 下的 $\text{train}(m)$ 和 $\text{val}(m)$ |
| `train_sizes=np.linspace(0.1, 1.0, 10)` | 10 个不同的训练集大小 |
| `validation_curve(model, X, y, param_name, param_range)` | 不同 $\lambda$ 下的训练/验证曲线 |
| `train_scores.mean(axis=1)` | $\text{train}(m)$（CV 平均） |
| `val_scores.mean(axis=1)` | $\text{val}(m)$（CV 平均） |
| `max_depth` 作为 `param_range` | 模型复杂度的代理参数 |

### 1. learning_curve 学习曲线

运行 1. learning_curve 学习曲线 的代码，观察算法在该环节的行为。

In [ ]:
print("=" * 60)
print("1. learning_curve - 训练集大小 vs 性能")
print("=" * 60)

model = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000, random_state=42))

train_sizes, train_scores, val_scores = learning_curve(
    model, X, y,
    train_sizes=np.linspace(0.1, 1.0, 10),
    cv=5, scoring='accuracy', random_state=42, n_jobs=-1
)

print(f"{'训练集大小':>10} {'训练准确率':>10} {'验证准确率':>10} {'差距':>8}")
print("-" * 45)
for size, t_mean, v_mean, t_std, v_std in zip(
    train_sizes,
    train_scores.mean(axis=1),
# --- 继续 ---
    val_scores.mean(axis=1),
    train_scores.std(axis=1),
    val_scores.std(axis=1)
):
    gap = t_mean - v_mean
# --- 输出结果 ---
    print(f"  {size:>8} {t_mean:>10.4f} {v_mean:>10.4f} {gap:>8.4f}")

# 诊断
final_train = train_scores.mean(axis=1)[-1]
final_val = val_scores.mean(axis=1)[-1]
gap_final = final_train - final_val

print(f"\n诊断结果:")
print(f"  最终训练准确率: {final_train:.4f}")
print(f"  最终验证准确率: {final_val:.4f}")
print(f"  训练-验证差距: {gap_final:.4f}")

if gap_final > 0.05 and final_val < 0.85:
    print("  -> 过拟合: 模型在训练集上表现好但泛化差，增加数据或简化模型")
elif final_val < 0.7 and final_train < 0.75:
    print("  -> 欠拟合: 模型容量不足，增加特征或使用更复杂的模型")
else:
# --- 输出结果 ---
    print("  -> 模型状态良好")

### 2. 不同复杂度模型的学习曲线对比

对比不同模型或配置的性能差异。

In [ ]:
print("\n" + "=" * 60)
print("2. 不同复杂度模型的学习曲线对比")
print("=" * 60)

models_complexity = {
    '逻辑回归(简单)': make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000, random_state=42)),
    '决策树(深度3)': DecisionTreeClassifier(max_depth=3, random_state=42),
    '决策树(无限制)': DecisionTreeClassifier(max_depth=None, random_state=42),
    '随机森林(100棵)': RandomForestClassifier(n_estimators=100, random_state=42),
# --- 继续 ---
}

print(f"{'模型':<18} {'训练准确率':>10} {'验证准确率':>10} {'差距':>8} {'状态':>8}")
print("-" * 60)

for name, m in models_complexity.items():
    ts, tr_s, va_s = learning_curve(
        m, X, y,
        train_sizes=np.linspace(0.3, 1.0, 3),
        cv=5, scoring='accuracy', random_state=42
# --- 继续 ---
    )
    final_tr = tr_s.mean(axis=1)[-1]
    final_va = va_s.mean(axis=1)[-1]
    gap = final_tr - final_va

    if gap > 0.08:
        status = "过拟合"
    elif final_va < 0.7:
        status = "欠拟合"
    else:
# --- 赋值/配置 ---
        status = "良好"

    print(f"  {name:<16} {final_tr:>10.4f} {final_va:>10.4f} {gap:>8.4f} {status:>8}")

### 3. validation_curve 验证曲线 (超参数搜索)

探索关键超参数对模型行为的影响。

In [ ]:
print("\n" + "=" * 60)
print("3. validation_curve - 超参数 vs 性能")
print("=" * 60)

# 决策树最大深度
tree = DecisionTreeClassifier(random_state=42)
depths = [1, 2, 3, 5, 8, 10, 15, 20, None]

train_scores_depth, val_scores_depth = validation_curve(
    tree, X, y,
    param_name='max_depth',
    param_range=depths,
    cv=5, scoring='accuracy'
# --- 继续 ---
)

print("决策树 max_depth 验证曲线:")
print(f"{'max_depth':>10} {'训练准确率':>10} {'验证准确率':>10} {'差距':>8}")
print("-" * 45)
for depth, t_mean, v_mean in zip(depths, train_scores_depth.mean(axis=1), val_scores_depth.mean(axis=1)):
    depth_str = str(depth) if depth is not None else "None"
# --- 赋值/配置 ---
    gap = t_mean - v_mean
    print(f"  {depth_str:>8} {t_mean:>10.4f} {v_mean:>10.4f} {gap:>8.4f}")

# 找到最佳深度
best_idx = np.argmax(val_scores_depth.mean(axis=1))
best_depth = depths[best_idx]
print(f"\n最佳max_depth: {best_depth} (验证准确率: {val_scores_depth.mean(axis=1)[best_idx]:.4f})")

### 4. 随机森林树数量验证曲线

运行 4. 随机森林树数量验证曲线 的代码，观察算法在该环节的行为。

In [ ]:
print("\n" + "=" * 60)
print("4. 随机森林 n_estimators 验证曲线")
print("=" * 60)

rf = RandomForestClassifier(random_state=42)
n_estimators_range = [5, 10, 25, 50, 100, 200]

train_scores_ne, val_scores_ne = validation_curve(
    rf, X, y,
    param_name='n_estimators',
    param_range=n_estimators_range,
    cv=5, scoring='accuracy'
# --- 继续 ---
)

print(f"{'n_estimators':>12} {'训练准确率':>10} {'验证准确率':>10} {'差距':>8}")
print("-" * 45)
for n_est, t_mean, v_mean in zip(
    n_estimators_range,
    train_scores_ne.mean(axis=1),
# --- 继续 ---
    val_scores_ne.mean(axis=1)
):
    gap = t_mean - v_mean
    print(f"  {n_est:>10} {t_mean:>10.4f} {v_mean:>10.4f} {gap:>8.4f}")

### 5. 综合诊断

运行 5. 综合诊断 的代码，观察算法在该环节的行为。

In [ ]:
print("\n" + "=" * 60)
print("5. 过拟合/欠拟合诊断总结")
print("=" * 60)

diagnostics = {
    '逻辑回归': {'train': 0.82, 'val': 0.80},
    '决策树(浅)': {'train': 0.85, 'val': 0.82},
    '决策树(深)': {'train': 1.00, 'val': 0.75},
    '随机森林': {'train': 0.95, 'val': 0.88},
# --- 继续 ---
}

print(f"{'模型':<14} {'训练':>6} {'验证':>6} {'差距':>6} {'诊断':<12}")
print("-" * 50)
for name, scores in diagnostics.items():
    gap = scores['train'] - scores['val']
    if gap > 0.15:
# --- 赋值/配置 ---
        diag = "严重过拟合"
    elif gap > 0.05:
        diag = "轻微过拟合"
    elif scores['val'] < 0.7:
        diag = "欠拟合"
# --- 继续 ---
    else:
        diag = "良好"
    print(f"  {name:<12} {scores['train']:>6.2f} {scores['val']:>6.2f} {gap:>6.2f} {diag}")

print("""
诊断规则:
  过拟合信号:
    - 训练准确率 >> 验证准确率 (差距大)
    - 训练准确率接近1.0，验证准确率明显低
# --- 继续 ---
    - 增加数据量后验证准确率仍在提升
  欠拟合信号:
    - 训练和验证准确率都很低
    - 增加数据量后性能不再提升
  解决方案:
# --- 继续 ---
    过拟合: 增加数据、正则化、简化模型、特征选择、Dropout
    欠拟合: 增加特征、使用更复杂模型、减少正则化、集成方法
""")

## 关键代码解释

```python
from sklearn.model_selection import learning_curve
train_sizes, train_scores, val_scores = learning_curve(
    model, X, y, train_sizes=np.linspace(0.1, 1.0, 10), cv=5)
```

## 注意事项

1. 训练和验证曲线趋于收敛时，增加数据不再有帮助
2. 两条线差距大 = 过拟合
3. 两条线都低 = 欠拟合

## 总结与延伸

以上代码展示了 **学习曲线** 的完整流程。

**解读要点**：
- 关注 **ROC 曲线** 下的面积（AUC）
- 观察 **交叉验证** 各折的方差
- 对比不同评估指标的适用场景

**进一步思考**：尝试修改关键参数（如正则化强度、学习率、树深度等），观察结果如何变化。

### 延伸阅读

- **验证曲线**：固定数据量，看超参数的影响
- **偏差-方差分解**：过拟合 = 高方差，欠拟合 = 高偏差
- **Early Stopping**：基于验证曲线的最佳停止点
